# 10 Group errors and residuals on the scored year (C-03 test, C-04)

Written by Yiling (2026-11-15). Uses the BD-07 predictions: **test** (2024) once the one-time test has run, otherwise the **dry run** (2023). Thresholds come from the frozen spec (`reports/bd07_frozen_spec.json`) and are not re-chosen. Errors are relative to the historical decision label. Groups with n < `MIN_GROUP_N` are not reported. Judge codes are anonymous; residuals are predictive, not causal.

In [1]:
import sys; sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from src import config
assert config.ROOT == Path.cwd().resolve().parent, "Run from the notebooks directory."
(config.REPORTS / "figures").mkdir(exist_ok=True)
import json
from src import explain
from src.fairness import group_metrics, residuals
LABEL = explain.available_label()
print("Using:", LABEL)
spec = json.loads((config.REPORTS / "bd07_frozen_spec.json").read_text())
pred = pd.read_parquet(config.DATA_PROCESSED / f"predictions_{LABEL}.parquet")
pred["split"] = "scored"
rows = explain.scored_rows(LABEL)
JUDGE = config.JUDGE_COL_FINAL if LABEL == "test" else "judge_id"
meta = rows[[config.ID_COL, "county_name", JUDGE, *config.FAIRNESS_GROUPS]].rename(columns={JUDGE: "judge_code"})
cuts = pd.read_csv(config.REPORTS / f"bd07_{LABEL}_confusion.csv").set_index(["model", "feature_set"])["threshold"]
print(cuts)

Using: test


model  feature_set     
xgb    CASE                0.352014
       CASE_COURT          0.341572
       CASE_COURT_JUDGE    0.351024
logit  CASE                0.341424
       CASE_COURT          0.340640
       CASE_COURT_JUDGE    0.343132
Name: threshold, dtype: float64


## C-03 Group errors and calibration

In [2]:
tables = []
for (model, fs), t in cuts.items():
    part = pred.loc[pred.model.eq(model) & pred.feature_set.eq(fs)]
    for g in config.FAIRNESS_GROUPS:
        tables.append(group_metrics(part, meta, g, t, split="scored"))
group_table = pd.concat(tables, ignore_index=True)
group_table.to_csv(config.REPORTS / f"c03_{LABEL}_group_metrics.csv", index=False)
show = ["model", "feature_set", "group", "n", "n_pos", "base_rate", "mean_pred", "calibration_gap", "pred_pos_rate", "fpr", "fnr", "roc_auc"]
group_table.loc[group_table.group_col.eq("race") & group_table.model.eq("xgb"), show].round(4)

,model,feature_set,group,n,n_pos,base_rate,mean_pred,calibration_gap,pred_pos_rate,fpr,fnr,roc_auc
0,xgb,CASE,American Indian/Alaskan Native,445,33,0.0742,0.0619,-0.0122,0.0629,0.0218,NaN,NaN
1,xgb,CASE,Asian/Pacific Islander,8433,754,0.0894,0.0709,-0.0186,0.0658,0.0275,0.5438,0.9375
2,xgb,CASE,Black,76927,10171,0.1322,0.1301,-0.0022,0.1433,0.0642,0.3375,0.9295
3,xgb,CASE,Unknown,688,106,0.1541,0.1195,-0.0346,0.1308,0.0515,0.4340,0.9368
4,xgb,CASE,White,49318,5276,0.1070,0.0939,-0.0131,0.0943,0.0417,0.4665,0.9283
16,xgb,CASE_COURT,American Indian/Alaskan Native,445,33,0.0742,0.0599,-0.0142,0.0607,0.0194,NaN,NaN
17,xgb,CASE_COURT,Asian/Pacific Islander,8433,754,0.0894,0.0728,-0.0166,0.0695,0.0275,0.5027,0.9424
18,xgb,CASE_COURT,Black,76927,10171,0.1322,0.1290,-0.0032,0.1439,0.0635,0.3289,0.9333
19,xgb,CASE_COURT,Unknown,688,106,0.1541,0.1236,-0.0305,0.1483,0.0567,0.3491,0.9411
20,xgb,CASE_COURT,White,49318,5276,0.1070,0.0950,-0.0120,0.0973,0.0425,0.4445,0.9324


In [3]:
x = group_table.loc[group_table.model.eq("xgb")].set_index(["group_col", "group", "feature_set"])
cols = ["calibration_gap", "pred_pos_rate", "fpr", "fnr"]
change = x.xs("CASE_COURT_JUDGE", level="feature_set")[cols] - x.xs("CASE", level="feature_set")[cols]
change.to_csv(config.REPORTS / f"c03_{LABEL}_judge_change.csv")
change.round(4)

calibration_gap  pred_pos_rate  \
group_col group                                                            
race      American Indian/Alaskan Native          -0.0012         0.0000   
          Asian/Pacific Islander                   0.0040         0.0057   
          Black                                    0.0001        -0.0023   
          Unknown                                  0.0029         0.0058   
          White                                    0.0025         0.0036   
ethnicity Hispanic                                 0.0016         0.0011   
          Non Hispanic                             0.0009        -0.0001   
          Unknown                                  0.0086         0.0103   
gender    Female                                  -0.0002         0.0006   
          Male                                     0.0016         0.0003   
age_group 18-24                                    0.0002        -0.0071   
          25-34                                    0.0012        -0.0007   
          35-44                                    0.0027         0.0042   
          45-54                                    0.0015         0.0049   
          55+                                      0.0003         0.0045   
          <18                                     -0.0090        -0.0230   

                                             fpr     fnr  
group_col group                                           
race      American Indian/Alaskan Native -0.0024     NaN  
          Asian/Pacific Islander         -0.0001 -0.0650  
          Black                          -0.0039 -0.0082  
          Unknown                        -0.0034 -0.0566  
          White                          -0.0000 -0.0341  
ethnicity Hispanic                       -0.0011 -0.0179  
          Non Hispanic                   -0.0028 -0.0195  
          Unknown                        -0.0058 -0.0714  
gender    Female                         -0.0014 -0.0376  
          Male                           -0.0025 -0.0179  
age_group 18-24                          -0.0074  0.0049  
          25-34                          -0.0024 -0.0125  
          35-44                           0.0004 -0.0307  
          45-54                           0.0005 -0.0404  
          55+                            -0.0005 -0.0517  
          <18                            -0.0319  0.0087

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
race = group_table.loc[group_table.group_col.eq("race") & group_table.model.eq("xgb")
                       & group_table.feature_set.isin(["CASE", "CASE_COURT_JUDGE"])]
order = race.loc[race.feature_set.eq("CASE")].sort_values("n", ascending=False)["group"].tolist()
for ax, metric, title in zip(axes, ["fpr", "fnr"], ["False positive rate", "False negative rate"]):
    for k, (fs, marker) in enumerate([("CASE", "o"), ("CASE_COURT_JUDGE", "s")]):
        r = race.loc[race.feature_set.eq(fs)].set_index("group").reindex(order)
        ax.scatter(r[metric], np.arange(len(order)) + (k - 0.5) * 0.25, marker=marker, label=fs)
    ax.set_title(title); ax.set_yticks(range(len(order))); ax.set_yticklabels(order); ax.grid(axis="x", alpha=0.3)
axes[0].legend(frameon=False, loc="lower right")
fig.suptitle(f"XGBoost, {LABEL}, frozen threshold (relative to historical decisions)")
fig.tight_layout(); fig.savefig(config.REPORTS / "figures" / f"c03_{LABEL}_race_errors.png", dpi=200); plt.close(fig)

## C-05 Disparity analysis on the scored year (XGBoost `CASE_COURT_JUDGE`)

Same five steps as notebook 08. The common threshold comes from the frozen spec; the Black-group threshold is **reused** from `reports/c05_group_threshold_valid.csv` (chosen on 2023), not searched again.

In [5]:
from src.fairness import disparity_rates, rate_comparison
FINAL = ("xgb", "CASE_COURT_JUDGE")
T = float(cuts[FINAL])
final_pred = pred.loc[pred.model.eq(FINAL[0]) & pred.feature_set.eq(FINAL[1])]
race_meta = meta[[config.ID_COL, "race"]]

rates_race = disparity_rates(final_pred, race_meta, "race", T, split="scored")
rates_race.to_csv(config.REPORTS / f"c05_{LABEL}_rates_by_race.csv", index=False)
display(rates_race.set_index("group")[["n", "share_pos", "TP", "FP", "FN", "TN", "flag_rate", "tpr", "fpr", "precision"]].round(3))
compare = rate_comparison(rates_race, "Black", "White")
compare.to_csv(config.REPORTS / f"c05_{LABEL}_black_white_comparison.csv", index=False)
display(compare.round(3))

# Reuse the 2023 group threshold, once
valid_thr = pd.read_csv(config.REPORTS / "c05_group_threshold_valid.csv")
T_BLACK = float(valid_thr.loc[valid_thr.scenario.eq("Black threshold raised") & valid_thr.group.eq("Black"), "threshold"].iloc[0])
adjusted = disparity_rates(final_pred, race_meta, "race", T, split="scored", thresholds_by_group={"Black": T_BLACK})
cols = ["threshold", "flag_rate", "tpr", "fpr", "precision"]
threshold_table = pd.concat({"common threshold": rates_race.set_index("group").loc[["Black", "White"], cols],
                             "Black threshold raised (from 2023)": adjusted.set_index("group").loc[["Black", "White"], cols]},
                            names=["scenario"])
threshold_table.to_csv(config.REPORTS / f"c05_{LABEL}_group_threshold.csv")
print(f"Common threshold {T:.3f}; Black-group threshold reused from 2023: {T_BLACK:.3f}")
display(threshold_table.round(3))

# Course-style bar chart for the deck
rates, labels = ["flag_rate", "tpr", "fpr", "precision"], ["Flag rate", "TPR", "FPR", "Precision"]
r = rates_race.set_index("group"); x = np.arange(len(rates))
fig, ax = plt.subplots(figsize=(7, 3.8))
for i, (grp, color) in enumerate([("Black", "#4E2A84"), ("White", "#D86018")]):
    vals = r.loc[grp, rates].astype(float)
    ax.bar(x + (i - 0.5) * 0.38, vals, 0.38, color=color, label=f"{grp} (n = {int(r.loc[grp, 'n']):,})")
    for xi, v in zip(x, vals):
        ax.text(xi + (i - 0.5) * 0.38, v + 0.015, f"{v:.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0, 1.0); ax.legend(frameon=False)
ax.set_title(f"Same model, same threshold, two groups ({LABEL}, relative to historical decisions)", fontsize=10)
fig.tight_layout(); fig.savefig(config.REPORTS / "figures" / f"c05_{LABEL}_rates_by_race.png", dpi=200); plt.close(fig)
print(f"Saved reports/figures/c05_{LABEL}_rates_by_race.png")

,n,share_pos,TP,FP,FN,TN,flag_rate,tpr,fpr,precision
group,,,,,,,,,,
American Indian/Alaskan Native,445,0.074,20,8,13,404,0.063,NaN,0.019,NaN
Asian/Pacific Islander,8433,0.089,393,210,361,7469,0.072,0.521,0.027,0.652
Black,76927,0.132,6821,4024,3350,62732,0.141,0.671,0.060,0.629
Unknown,688,0.154,66,28,40,554,0.137,0.623,0.048,NaN
White,49318,0.107,2995,1836,2281,42206,0.098,0.568,0.042,0.620


,rate,Black,White,gap,ratio,four_fifths
0,flag_rate,0.141,0.098,0.043,0.695,False
1,tpr,0.671,0.568,0.103,0.846,NaN
2,fpr,0.060,0.042,0.019,0.692,NaN
3,precision,0.629,0.620,0.009,0.986,NaN


Common threshold 0.351; Black-group threshold reused from 2023: 0.488


threshold  flag_rate    tpr    fpr  \
scenario                           group                                       
common threshold                   Black      0.351      0.141  0.671  0.060   
                                   White      0.351      0.098  0.568  0.042   
Black threshold raised (from 2023) Black      0.488      0.107  0.563  0.038   
                                   White      0.351      0.098  0.568  0.042   

                                          precision  
scenario                           group             
common threshold                   Black      0.629  
                                   White      0.620  
Black threshold raised (from 2023) Black      0.695  
                                   White      0.620

Saved reports/figures/c05_test_rates_by_race.png


## C-04 Residuals by county and by anonymous judge code (XGBoost CASE)

Observed rate minus mean predicted probability from the case-and-record model. Positive = more bail/remand than predicted for those cases.

In [6]:
part = pred.loc[pred.model.eq("xgb") & pred.feature_set.eq("CASE")]
county = residuals(part, meta, "county_name", split="scored")
judge = residuals(part, meta, "judge_code", split="scored")
coded = judge.loc[judge["group"].str.startswith("J")].copy()
county.to_csv(config.REPORTS / f"c04_{LABEL}_county_residuals.csv", index=False)
coded.to_csv(config.REPORTS / f"c04_{LABEL}_judge_residuals.csv", index=False)
summary = pd.DataFrame({
    "coded_judges_reported": [len(coded)],
    "share_ci_above_0": [(coded["ci_low"] > 0).mean()],
    "share_ci_below_0": [(coded["ci_high"] < 0).mean()],
    "residual_p10": [coded["residual"].quantile(0.1)], "residual_p90": [coded["residual"].quantile(0.9)],
})
summary.to_csv(config.REPORTS / f"c04_{LABEL}_judge_residual_summary.csv", index=False)
display(county.round(4)); summary.round(4)

,model,feature_set,by,group,n,observed_rate,mean_pred,residual,ci_low,ci_high
0,xgb,CASE,county_name,Bronx,23696,0.1026,0.1225,-0.0199,-0.0229,-0.0171
1,xgb,CASE,county_name,Kings,44888,0.0942,0.1117,-0.0174,-0.0196,-0.0154
2,xgb,CASE,county_name,New York,28506,0.1449,0.1173,0.0276,0.0245,0.0309
3,xgb,CASE,county_name,Queens,31815,0.1513,0.1070,0.0444,0.0414,0.0474
4,xgb,CASE,county_name,Richmond,6916,0.1067,0.0989,0.0078,0.0020,0.0134


,coded_judges_reported,share_ci_above_0,share_ci_below_0,residual_p10,residual_p90
0,119,0.3193,0.1933,-0.038,0.0797


In [7]:
r = coded.sort_values("residual").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(np.arange(len(r)), r["residual"], yerr=[r["residual"] - r["ci_low"], r["ci_high"] - r["residual"]],
            fmt="o", ms=2.5, lw=0.6, color="#4C72B0", ecolor="#9DB4D9")
ax.axhline(0, color="k", lw=0.7)
ax.set_xlabel("Anonymous judge codes (sorted)"); ax.set_ylabel("Observed − predicted")
ax.set_title(f"Judge residuals, XGBoost CASE, {LABEL} (95% bootstrap intervals)")
fig.tight_layout(); fig.savefig(config.REPORTS / "figures" / f"c04_{LABEL}_judge_residuals.png", dpi=200); plt.close(fig)
print("Done. No judge names are used anywhere.")

Done. No judge names are used anywhere.
